# Análise de RX de Tórax com PDI

**Versão 0.1 — fundamentos, anonimização e análise quantitativa**

> Projeto acadêmico. Não substitui avaliação médica.


## Bases públicas previstas

O projeto utilizará as bases em etapas diferentes:

1. **Montgomery County CXR Set:** segmentação pulmonar e máscaras de referência;
2. **Shenzhen Hospital CXR Set:** teste de generalização;
3. **CheXpert:** classificação exploratória de achados;
4. **MIMIC-CXR-JPG:** estudos avançados em larga escala.

Os links oficiais e as regras de uso estão em `docs/DATASETS.md`.


## 1. Preparação do ambiente

Quando o notebook estiver no GitHub, o primeiro bloco clona o repositório. Durante o desenvolvimento local, ele usa a pasta já disponível.


In [ ]:
!pip -q install numpy opencv-python-headless matplotlib pandas pydicom Pillow scikit-image pytest

import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/suselen/segmentacao-rx-torax.git'
REPO_DIR = Path('/content/segmentacao-rx-torax')

if 'google.colab' in sys.modules:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print('Pasta de trabalho:', Path.cwd())


## 2. Importações


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.io_utils import read_image
from src.anonymization import default_corner_masks, mask_pixel_regions
from src.preprocessing import preprocess
from src.measurements import intensity_statistics
from src.segmentation import exploratory_lung_candidate_mask
from src.visualization import show_images, plot_histogram, overlay_mask
from src.report import metrics_table, technical_draft


## 3. Envio da radiografia

Envie PNG, JPG, TIFF ou DICOM. Use somente imagem pública, sintética ou previamente autorizada.


In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Nenhum arquivo foi enviado.')

input_name = next(iter(uploaded))
image, dicom_dataset = read_image(input_name)

print('Arquivo:', input_name)
print('Dimensão:', image.shape)
print('Tipo:', image.dtype)


## 4. Mascaramento de textos identificadores

O bloco abaixo cobre os quatro cantos. Revise visualmente e ajuste os retângulos. Cada retângulo usa `(x, y, largura, altura)`.


In [ ]:
rectangles = list(default_corner_masks(image, fraction=0.12))

# Exemplo de máscara adicional:
# rectangles.append((20, 30, 300, 80))

image_masked = mask_pixel_regions(image, rectangles, value=0)

show_images({
    'Original recebida': image,
    'Após mascaramento': image_masked,
}, cols=2, figsize=(12, 6));


## 5. CLAHE, filtros e detecção de bordas


In [ ]:
processed = preprocess(image_masked)

show_images({
    'Original anonimizada': processed.original,
    'CLAHE': processed.clahe,
    'Gaussiano': processed.gaussian,
    'Mediana': processed.median,
    'Bilateral': processed.bilateral,
    'Sobel': processed.sobel,
    'Canny': processed.canny,
}, cols=3, figsize=(15, 14));


## 6. Histograma e estatísticas


In [ ]:
stats = intensity_statistics(processed.clahe)
display(metrics_table(stats).round(4))
plot_histogram(processed.clahe);


## 7. Linha de base exploratória para candidatos pulmonares

Este resultado é apenas uma demonstração de limiarização, morfologia e componentes conexos. Ele ainda não é uma segmentação anatômica validada.


In [ ]:
candidate_mask = exploratory_lung_candidate_mask(processed.clahe)
overlay = overlay_mask(processed.original, candidate_mask)

show_images({
    'Máscara exploratória': candidate_mask,
    'Sobreposição': overlay,
}, cols=2, figsize=(12, 7));


## 8. Descrição técnica quantitativa


In [ ]:
edge_percentage = 100.0 * np.count_nonzero(processed.canny) / processed.canny.size

draft = technical_draft(
    image_shape=processed.original.shape,
    stats=stats,
    edge_pixel_percentage=edge_percentage,
    segmentation_status='linha de base exploratória, ainda não validada',
)

print(draft)


## 9. Próximas etapas

1. selecionar uma base pública com máscaras pulmonares;
2. padronizar tamanho e intensidade;
3. carregar máscara de referência;
4. calcular Dice e IoU;
5. implementar segmentação pulmonar validada;
6. adicionar silhueta cardíaca e índice cardiotorácico;
7. localizar ângulos costofrênicos;
8. criar página final de relatório.
